# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
required_columns = [
    "content_id",
    "trend_pct",
    "ctr",
    "avg_position",
    "engagement_rate"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Required columns check")
print("======================")

if len(missing_columns) == 0:
    print("PASS: All required columns are available.")
else:
    print("Missing columns:", missing_columns)

Required columns check
PASS: All required columns are available.


In [14]:
import pandas as pd
import os

# Load the dataset again
file_path = "/content/flyrank-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print("Data loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Data loaded successfully
Rows: 30000
Columns: 44


My rule:

I will prioritize content items that show stronger signs of declining search performance. A higher action score means the page should be reviewed earlier by the SEO/content team. The score is a prioritization tool, not an automatic decision.

Reason codes:

- HIGH_DECLINE: Large negative trend percentage.
- LOW_CTR: Click-through rate is relatively low.
- LOW_POSITION: Average search position is relatively weak.
- LOW_ENGAGEMENT: Engagement rate is relatively low.
- MULTIPLE_SIGNALS: More than one warning signal is present.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
import numpy as np
import os

score_df = df[
    [
        "content_id",
        "client_id",
        "trend_pct",
        "ctr",
        "avg_position",
        "engagement_rate"
    ]
].copy()

# Fill missing values
score_df["trend_pct"] = score_df["trend_pct"].fillna(0)
score_df["ctr"] = score_df["ctr"].fillna(score_df["ctr"].median())
score_df["avg_position"] = score_df["avg_position"].fillna(
    score_df["avg_position"].median()
)
score_df["engagement_rate"] = score_df["engagement_rate"].fillna(
    score_df["engagement_rate"].median()
)

# Calculate baseline action score
score_df["action_score"] = (
    (-score_df["trend_pct"]).clip(lower=0)
    + ((score_df["avg_position"] - 10).clip(lower=0) * 2)
    + ((0.5 - score_df["ctr"]).clip(lower=0) * 20)
    + ((0.5 - score_df["engagement_rate"]).clip(lower=0) * 10)
)

# Rank highest priority first
score_df = score_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

score_df["rank"] = np.arange(1, len(score_df) + 1)

# Save output
output_dir = "/content/flyrank-starter/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

score_df.to_csv(output_path, index=False)

print("Ranked queue created successfully")
print("Rows:", len(score_df))
print("Output:", output_path)

display(score_df.head(20))

Ranked queue created successfully
Rows: 30000
Output: /content/flyrank-starter/work/outputs/baseline_action_score.csv


,content_id,client_id,trend_pct,ctr,avg_position,engagement_rate,action_score,rank
0,content_661e1745db72,client_e29c9c180c,0.0,0.0,245.0,0.0,485.0,1
1,content_23f1cc8851a9,client_e29c9c180c,0.0,0.0,184.0,0.0,363.0,2
2,content_13bbd72aea33,client_e29c9c180c,-100.0,0.0,118.0,0.0,331.0,3
3,content_7275a6a3a8eb,client_e29c9c180c,0.0,0.0,165.5,0.0,326.0,4
4,content_71a31b831092,client_e29c9c180c,0.0,0.0,161.0,0.0,317.0,5
5,content_638236e8066e,client_e29c9c180c,-100.0,0.0,98.0,0.0,291.0,6
6,content_42c7c72b8391,client_e29c9c180c,0.0,0.0,145.5,0.0,286.0,7
7,content_cb6c7d58c0bc,client_e29c9c180c,0.0,0.0,144.5,0.0,284.0,8
8,content_f616ca0ec5ea,client_e29c9c180c,-100.0,0.0,94.0,0.0,283.0,9
9,content_692fda8c52bd,client_e29c9c180c,0.0,0.0,142.0,0.0,279.0,10


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# ML-07 Section 3: Top-20 Review

top20 = score_df.head(20).copy()

def get_reason(row):
    reasons = []

    if row["trend_pct"] < -20:
        reasons.append("HIGH_DECLINE")

    if row["ctr"] < 0.5:
        reasons.append("LOW_CTR")

    if row["avg_position"] > 10:
        reasons.append("LOW_POSITION")

    if row["engagement_rate"] < 0.5:
        reasons.append("LOW_ENGAGEMENT")

    if len(reasons) >= 2:
        reasons.append("MULTIPLE_SIGNALS")

    return ", ".join(reasons) if reasons else "REVIEW"


top20["reason_code"] = top20.apply(get_reason, axis=1)

top20["action"] = "Review content"

top20["confidence_note"] = (
    "Directional prioritization based on observed signals."
)

top20["what_would_make_it_wrong"] = (
    "Missing data, unusual content context, or external search changes."
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "action_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,1,content_661e1745db72,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",485.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
1,2,content_23f1cc8851a9,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",363.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
2,3,content_13bbd72aea33,Review content,"HIGH_DECLINE, LOW_CTR, LOW_POSITION, LOW_ENGAG...",331.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
3,4,content_7275a6a3a8eb,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",326.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
4,5,content_71a31b831092,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",317.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
5,6,content_638236e8066e,Review content,"HIGH_DECLINE, LOW_CTR, LOW_POSITION, LOW_ENGAG...",291.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
6,7,content_42c7c72b8391,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",286.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
7,8,content_cb6c7d58c0bc,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",284.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
8,9,content_f616ca0ec5ea,Review content,"HIGH_DECLINE, LOW_CTR, LOW_POSITION, LOW_ENGAG...",283.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."
9,10,content_692fda8c52bd,Review content,"LOW_CTR, LOW_POSITION, LOW_ENGAGEMENT, MULTIPL...",279.0,Directional prioritization based on observed s...,"Missing data, unusual content context, or exte..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# ML-07 Section 4: Weak picks + leakage check

print("WEAK PICKS")
print("=" * 40)

# Show the 10 lowest-priority items
weak_picks = score_df.tail(10)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "trend_pct",
            "ctr",
            "avg_position",
            "engagement_rate"
        ]
    ]
)

print("\nLEAKAGE CHECK")
print("=" * 40)

# Fields that should not be used as future/product information
leakage_fields = [
    "trend_direction",
    "position_tier",
    "impression_tier"
]

for col in leakage_fields:
    print(f"{col}:", "NOT USED" if col not in score_df.columns else "CHECK NEEDED")

print("\nBaseline score uses observed trend_pct as a prioritization signal.")
print("No product flags or future-window features were intentionally added.")

WEAK PICKS


,rank,content_id,action_score,trend_pct,ctr,avg_position,engagement_rate
29990,29991,content_9af722104e41,0.0,1.6,0.90,8.9,2.78
29991,29992,content_39ddc9753eb4,0.0,8.3,1.61,7.1,1.96
29992,29993,content_227ae37684d3,0.0,1.7,0.59,3.5,2.00
29993,29994,content_31eef8eeedc1,0.0,24.5,0.51,4.1,0.90
29994,29995,content_f89dd7f9e607,0.0,25.0,11.11,6.6,100.00
29995,29996,content_c06d9718f701,0.0,433.3,4.35,4.9,50.00
29996,29997,content_5495bd237770,0.0,34.4,0.51,7.7,3.85
29997,29998,content_da6a103cafcf,0.0,35.5,0.51,8.9,15.38
29998,29999,content_c4b0cd44a056,0.0,16.3,1.23,2.8,1.31
29999,30000,content_0c87fe8d58d9,0.0,4.4,0.53,4.8,4.17



LEAKAGE CHECK
trend_direction: NOT USED
position_tier: NOT USED
impression_tier: NOT USED

Baseline score uses observed trend_pct as a prioritization signal.
No product flags or future-window features were intentionally added.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [20]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-starter

fatal: destination path '/content/flyrank-starter' already exists and is not an empty directory.


In [21]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-starter

fatal: destination path '/content/flyrank-starter' already exists and is not an empty directory.


In [22]:
import os

print("Repository exists:",
      os.path.exists("/content/flyrank-starter"))

print("\nFolders:")
print(os.listdir("/content/flyrank-starter"))

Repository exists: True

Folders:
['work', 'LICENSE', 'docs', '.gitignore', '.github', 'README.md', 'skills', '.git', 'outputs', 'AGENTS.md', 'GUIDE.md', 'submission', 'SETUP.md', 'CLAUDE.md', 'requirements.txt', 'DATA_USE.md', 'data', 'scripts', 'notebooks']


In [23]:
import pandas as pd

file_path = "/content/flyrank-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print("Data loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Data loaded successfully
Rows: 30000
Columns: 44
